# K=3 exploratory fit on the 98-benchmark data: two basins, six chains each
Identified eta r̂ is **1.676** pooled but **1.004 / 1.003** inside each basin, on 15 divergences / 36,000 and no straggler chain.
Basin A leads basin B by **32.2 nats**; the two are a partial rotation of the whole frame, not an axis relabelling.
The **4** new benchmarks that Epoch's own ECI uses (4% of rows) carry **28.0%** of the squared loading displacement.

In [1]:
import sys, json, itertools
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "fit.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import xarray as xr
import arviz as az
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from data import load_eci_data
from analysis import mirt_identified_rhat

FIT = "mirt_humanprior_lineageprior_floors"
TRACE = ROOT / f"results/{FIT}/trace_mirt_k3_humanprior_lineageprior_floors.nc"
PLOTS = ROOT / "plots/k3_basin_deepdive"
PLOTS.mkdir(parents=True, exist_ok=True)
THIN = 5

C = dict(blue="#0072B2", sky="#56B4E9", orange="#E69F00", verm="#D55E00",
         green="#009E73", pink="#CC79A7", gray="#999999", dark="#333333")

def show(fig, name):
    fig.update_layout(template="plotly_white", title=dict(x=0.5, font=dict(size=13)))
    fig.write_image(PLOTS / f"{name}.png", scale=2)
    fig.show()

# ── trace: only what the claims need, thinned; log_likelihood never touched ──
post = xr.open_dataset(TRACE, group="posterior")[["A", "theta", "D", "sigma_b", "tau_CD"]] \
         .isel(draw=slice(None, None, THIN)).load()
stats = xr.open_dataset(TRACE, group="sample_stats")[["logp", "diverging", "n_steps"]].load()

data = load_eci_data(include_all_benchmarks=True)
benches = data.blookup.sort_values("benchmark_idx")["benchmark"].tolist()
models = data.mlookup.sort_values("model_idx")["model"].tolist()
HUMANS = set(json.loads(post.attrs["mirt_human_order"]))

# ── basins: cluster chains on best-permutation loading correlation ───────────
Amed = np.median(post["A"].values, axis=1)          # (chain, bench, latent)
THmed = np.median(post["theta"].values, axis=1)     # (chain, model, latent)
NC, K = Amed.shape[0], Amed.shape[2]
PERMS = list(itertools.permutations(range(K)))

def match(c1, c2):
    """Best axis permutation of chain c2 onto c1, with the matched per-axis correlations."""
    M = np.corrcoef(Amed[c1].T, Amed[c2].T)[:K, K:]
    p = max(PERMS, key=lambda q: sum(M[i, q[i]] for i in range(K)))
    return list(p), np.array([M[i, p[i]] for i in range(K)])

# alignment score = WORST matched axis: within-basin ~0.999, across ~0.49, so 0.9 separates
ALIGN = np.array([[match(a, b)[1].min() for b in range(NC)] for a in range(NC)])
lab = -np.ones(NC, int)
for c in range(NC):
    if lab[c] < 0:
        lab[c] = lab.max() + 1
        lab[(lab < 0) & (ALIGN[c] >= 0.9)] = lab[c]
assert lab.max() == 1, f"expected 2 basins, got {lab.max() + 1}"

lp = stats["logp"].values
grp = sorted(range(2), key=lambda g: -lp[lab == g].mean())          # basin A = higher logp
BA = [c for c in range(NC) if lab[c] == grp[0]]
BB = [c for c in range(NC) if lab[c] == grp[1]]
GAP = lp[BB].mean() - lp[BA].mean()

# ── one common frame: every chain permuted onto the first basin-A chain ──────
REF = BA[0]
PERM = [match(REF, c)[0] for c in range(NC)]
def basin_mean(med, chains):
    return np.mean([med[c][:, PERM[c]] for c in chains], axis=0)

A_A, A_B = basin_mean(Amed, BA), basin_mean(Amed, BB)
TH_A, TH_B = basin_mean(THmed, BA), basin_mean(THmed, BB)

def top_b(Ax, k, n=3):
    return [benches[i] for i in np.argsort(-Ax[:, k])[:n]]

NAME = {"GBAEval": "Easy knowledge",                 # KeyError = the axes moved
        "VPCT": "Fluid / abstract",
        "FrontierMath Tier 4": "Hard math + science"}
AXIS = [NAME[top_b(A_A, k, 1)[0]] for k in range(K)]
LEG = [f"{AXIS[k]}<br><span style='font-size:11px'>{', '.join(top_b(A_A, k))}</span>" for k in range(K)]

# ── the 13 benchmarks the 2026-07-27/28 refresh added (85 -> 98) ─────────────
NEW_IN = ["EBR-bench", "GBAEval", "GDPval", "Surface Evolver Bench"]        # in Epoch's published ECI
NEW_OUT = ["AlgoTune", "BlueprintBench 2", "DeepSWE", "FrontierMath Tier 4 v1", "FrontierMath v1",
           "GDP.pdf", "MindCube", "ProofBench", "SpatialViz-Bench"]         # not in it
NEW = set(NEW_IN) | set(NEW_OUT)
assert NEW <= set(benches) and len(NEW) == 13
G_IN, G_OUT, G_OLD = "new / in Epoch ECI", "new / not in ECI", "pre-existing"
GROUP = [G_IN if b in NEW_IN else G_OUT if b in NEW_OUT else G_OLD for b in benches]
GCOL = {G_IN: C["verm"], G_OUT: C["orange"], G_OLD: C["gray"]}

DISP = np.abs(A_B - A_A).sum(axis=1)                 # per-benchmark loading displacement
DTH = np.abs(TH_B - TH_A).sum(axis=1)                # per-model ability displacement

print(f"{post.sizes['draw']} draws/chain x {NC} chains · {len(data.scores)} obs · "
      f"{data.n_models} models · {data.n_benchmarks} benchmarks")
print(f"basin A {BA} logp {lp[BA].mean():.1f} · basin B {BB} logp {lp[BB].mean():.1f} · gap {GAP:.1f} nats")
print(f"worst cross-basin alignment {ALIGN[np.ix_(BA, BB)].min():.3f} · "
      f"worst within-basin {min(ALIGN[np.ix_(g, g)].min() for g in (BA, BB)):.3f}")
print("axes:", AXIS)

600 draws/chain x 12 chains · 4447 obs · 765 models · 98 benchmarks
basin A [1, 3, 5, 6, 8, 10] logp 3508.4 · basin B [0, 2, 4, 7, 9, 11] logp 3476.2 · gap -32.2 nats
worst cross-basin alignment 0.482 · worst within-basin 0.999
axes: ['Easy knowledge', 'Fluid / abstract', 'Hard math + science']


### 1 · Twelve chains, two basins, split six-six.
Basin A sits **32.2 nats** above basin B. Every chain pair inside a basin matches at **0.999**; every cross-basin pair at **0.48**.

In [2]:
ORD = BA + BB
tick = [f"c{c}" for c in ORD]
m, (lo, hi) = lp.mean(axis=1), np.quantile(lp, [0.05, 0.95], axis=1)

fig = make_subplots(rows=1, cols=2, column_widths=[0.44, 0.56], horizontal_spacing=0.13,
                    subplot_titles=["Log posterior density per chain",
                                    "Chain-pair alignment (worst matched axis)"])
for tag, sel, colr in [(f"basin A ({len(BA)} chains)", BA, C["blue"]),
                       (f"basin B ({len(BB)} chains)", BB, C["verm"])]:
    fig.add_trace(go.Scatter(
        x=[f"c{c}" for c in sel], y=m[sel], mode="markers", name=tag,
        marker=dict(color=colr, size=11, symbol="diamond", line=dict(color="white", width=1)),
        error_y=dict(type="data", symmetric=False, array=hi[sel] - m[sel], arrayminus=m[sel] - lo[sel],
                     color=colr, thickness=1.3, width=5),
        hovertemplate="%{x}: %{y:.1f}<extra></extra>"), row=1, col=1)
    fig.add_trace(go.Scatter(x=tick, y=[lp[sel].mean()] * len(tick), mode="lines",
                             name=f"{tag.split(' (')[0]} mean", line=dict(color=colr, width=1, dash="dot"),
                             hoverinfo="skip"), row=1, col=1)
fig.add_annotation(x=tick[len(BA)], y=(lp[BA].mean() + lp[BB].mean()) / 2, text=f"<b>{GAP:.1f} nats</b>",
                   showarrow=True, arrowhead=2, ax=44, ay=0, font=dict(size=12, color=C["dark"]),
                   arrowcolor=C["dark"], row=1, col=1)

fig.add_trace(go.Heatmap(
    z=ALIGN[np.ix_(ORD, ORD)], x=tick, y=tick, zmin=0.4, zmax=1.0,
    colorscale=[[0, "#FFF7EC"], [0.5, C["orange"]], [1.0, C["blue"]]],
    colorbar=dict(title="worst matched<br>axis corr", thickness=12, len=0.78, x=1.005, y=0.44),
    hovertemplate="%{y} vs %{x}: %{z:.3f}<extra></extra>"), row=1, col=2)
for pos in (len(BA) - 0.5,):
    fig.add_vline(x=pos, line=dict(color=C["dark"], width=2), row=1, col=2)
    fig.add_hline(y=pos, line=dict(color=C["dark"], width=2), row=1, col=2)

fig.update_xaxes(categoryorder="array", categoryarray=tick, title_text="chain", row=1, col=1)
fig.update_xaxes(title_text="chain (basin A left, basin B right)", row=1, col=2)
fig.update_yaxes(title_text="mean logp (5-95% of draws)", row=1, col=1)
fig.update_yaxes(autorange="reversed", row=1, col=2)
fig.update_layout(title="Two basins: 6 chains each, 32 nats apart", height=470, width=1120,
                  legend=dict(orientation="h", y=-0.20), margin=dict(t=90, r=110))
show(fig, "p1_two_basins")

### 2 · Each basin is internally converged; only the pooled fit is not.
Identified r̂ falls from **1.68 / 1.68 / 1.66** pooled to **≤ 1.005** inside either basin. The pooled number is a mixture artefact.

In [3]:
RHAT = {tag: mirt_identified_rhat(az.InferenceData(posterior=post.isel(chain=sel)), data)
        for tag, sel in [("pooled (12 chains)", list(range(NC))), ("basin A (6)", BA), ("basin B (6)", BB)]}
keys = ["eta_max_rhat", "D_max_rhat", "sigma_b_max_rhat"]
lab3 = ["eta (max)", "D (max)", "sigma_b (max)"]

fig = go.Figure()
for (tag, r), colr in zip(RHAT.items(), [C["verm"], C["blue"], C["sky"]]):
    v = [float(r[k]) for k in keys]
    fig.add_trace(go.Bar(x=lab3, y=v, name=tag, marker_color=colr,
                         text=[f"{x:.3f}" for x in v], textposition="outside",
                         textfont=dict(size=11)))
fig.add_trace(go.Scatter(x=lab3, y=[1.01] * 3, mode="lines", name="r̂ = 1.01 (convergence bar)",
                         line=dict(color=C["dark"], width=1.5, dash="dot")))
fig.update_layout(title="Identified r̂: pooled versus within basin",
                  yaxis_title="r̂", yaxis_range=[1.0, 1.80], height=450, width=800,
                  legend=dict(orientation="h", y=1.10), bargap=0.28)
show(fig, "p2_rhat")

### 3 · The disagreement is not one axis: cross-basin correlation **0.49 / 0.81 / 0.91**.
Within a basin every axis reproduces at 0.999. Across basins the whole frame tilts, worst on the easy-knowledge axis.

In [4]:
def axis_corr(c1, c2):
    a1, a2 = Amed[c1][:, PERM[c1]], Amed[c2][:, PERM[c2]]
    return np.array([np.corrcoef(a1[:, k], a2[:, k])[0, 1] for k in range(K)])

within = np.mean([axis_corr(a, b) for g in (BA, BB) for a, b in itertools.combinations(g, 2)], axis=0)
across = np.mean([axis_corr(a, b) for a in BA for b in BB], axis=0)

fig = go.Figure()
for tag, v, colr, off in [("within a basin (mean over 30 pairs)", within, C["blue"], -0.13),
                          ("across basins (mean over 36 pairs)", across, C["verm"], 0.13)]:
    for k in range(K):
        fig.add_trace(go.Scatter(x=[k + off, k + off], y=[0, v[k]], mode="lines",
                                 line=dict(color=colr, width=2), showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=[k + off for k in range(K)], y=v, mode="markers+text", name=tag,
                             marker=dict(color=colr, size=15, line=dict(color="white", width=1)),
                             text=[f"{x:.2f}" for x in v], textposition="top center",
                             textfont=dict(size=11, color=colr),
                             hovertemplate="%{y:.3f}<extra></extra>"))
fig.update_xaxes(tickmode="array", tickvals=list(range(K)), ticktext=LEG, title_text="axis")
fig.update_layout(title="Loading-column correlation, within basin versus across basins",
                  yaxis=dict(title="Pearson r", range=[0, 1.15]), height=470, width=880,
                  legend=dict(orientation="h", y=1.12))
show(fig, "p3_axis_corr")

### 4 · Four benchmarks Epoch actually uses carry **28.0%** of the displacement.
They are 4% of the rows. The nine new benchmarks outside Epoch's ECI carry 11.9% on 9% of the rows; both FrontierMath v1 items barely move.

In [5]:
o = np.argsort(-DISP)
rank = {i: r for r, i in enumerate(o)}
sq = DISP ** 2

fig = make_subplots(rows=1, cols=2, column_widths=[0.76, 0.24], horizontal_spacing=0.09,
                    subplot_titles=["Loading displacement between basins, all 98 benchmarks",
                                    "Share of squared displacement"])
for g in (G_OLD, G_OUT, G_IN):
    idx = [i for i in o if GROUP[i] == g]
    fig.add_trace(go.Bar(x=[rank[i] for i in idx], y=DISP[idx], name=g, marker_color=GCOL[g],
                         width=0.85, text=[benches[i] for i in idx], hoverinfo="text+y",
                         hovertemplate="%{text}: %{y:.2f}<extra></extra>"), row=1, col=1)
newpos = sorted([i for i in range(len(benches)) if benches[i] in NEW], key=lambda i: rank[i])
fig.update_xaxes(tickmode="array", tickvals=[rank[i] for i in newpos],
                 ticktext=[f"<b>{benches[i]}</b>" if benches[i] in NEW_IN else benches[i] for i in newpos],
                 tickangle=-55, tickfont=dict(size=10),
                 title_text="98 benchmarks, sorted; ticks label the 13 added on 2026-07-27",
                 range=[-1, 98], row=1, col=1)
fv = [rank[benches.index(b)] for b in ("FrontierMath v1", "FrontierMath Tier 4 v1")]
fig.add_annotation(x=float(np.mean(fv)), y=1.05, text="FrontierMath v1 pair", showarrow=True, arrowhead=2,
                   ax=0, ay=-38, font=dict(size=11, color=C["dark"]), arrowcolor=C["dark"], row=1, col=1)
fig.update_yaxes(title_text="Σ<sub>k</sub> |A<sub>B</sub> − A<sub>A</sub>|", range=[0, 5.0], row=1, col=1)

for g in (G_IN, G_OUT, G_OLD):
    idx = [i for i in range(len(benches)) if GROUP[i] == g]
    pct = 100 * sq[idx].sum() / sq.sum()
    fig.add_trace(go.Bar(x=[g.replace(" / ", "<br>")], y=[pct], marker_color=GCOL[g], showlegend=False,
                         text=[f"{pct:.1f}%<br><span style='font-size:10px'>n={len(idx)}, "
                               f"mean {DISP[idx].mean():.2f}</span>"],
                         textposition="outside", textfont=dict(size=11),
                         hovertemplate="%{y:.1f}%<extra></extra>"), row=1, col=2)
fig.update_yaxes(title_text="% of Σ displacement²", range=[0, 78], row=1, col=2)
fig.update_layout(title="Where the two modes disagree: 4 new in-ECI benchmarks carry 28% of it",
                  height=560, width=1180, legend=dict(orientation="h", y=1.10),
                  margin=dict(b=150, t=110), bargap=0.15)
show(fig, "p4_displacement")

### 5 · What the two modes actually say.
GBAEval is the easy-knowledge axis in basin A and moves onto the fluid axis in basin B. VPCT and Cybench move the other way.

In [6]:
fig = make_subplots(rows=1, cols=K, horizontal_spacing=0.115,
                    subplot_titles=[f"{AXIS[k]} axis" for k in range(K)])
for col, k in enumerate(range(K), 1):
    sel = np.argsort(-np.maximum(A_A[:, k], A_B[:, k]))[:8][::-1]
    names = [f"<b>{benches[i]}</b>" if benches[i] in NEW else benches[i] for i in sel]
    for tag, vals, colr in [("basin A", A_A[sel, k], C["blue"]), ("basin B", A_B[sel, k], C["verm"])]:
        fig.add_trace(go.Bar(x=vals, y=names, orientation="h", name=tag, legendgroup=tag,
                             showlegend=col == 1, marker_color=colr,
                             hovertemplate="%{y}: %{x:.2f}<extra></extra>"), row=1, col=col)
    fig.update_xaxes(title_text="loading", range=[0, 1.05 * max(A_A[:, k].max(), A_B[:, k].max())],
                     row=1, col=col)
    fig.update_yaxes(tickfont=dict(size=10), row=1, col=col)
fig.add_trace(go.Bar(x=[None], y=[None], marker_color="rgba(0,0,0,0)",
                     name="bold = added in the 2026-07-27 refresh"), row=1, col=1)
fig.update_layout(title="Top 8 loadings per axis, in each basin",
                  height=470, width=1220, barmode="group", bargap=0.22,
                  legend=dict(orientation="h", y=1.13), margin=dict(t=110))
show(fig, "p5_mode_loadings")

### 6 · The modes are related by a partial rotation, not a relabelling.
A least-squares map `A_B ≈ A_A · M` reaches R² 0.71 with 32% of its mass off-diagonal; the pure relabelling (M = I) reaches only 0.54.

In [7]:
M, *_ = np.linalg.lstsq(A_A, A_B, rcond=None)
sse = lambda P: ((A_B - P) ** 2).sum()
sst = ((A_B - A_B.mean(axis=0)) ** 2).sum()
r2, r2_id = 1 - sse(A_A @ M) / sst, 1 - sse(A_A) / sst
off = (np.abs(M).sum() - np.abs(np.diag(M)).sum()) / np.abs(M).sum()

fig = go.Figure(go.Heatmap(
    z=M, x=[f"{AXIS[k]}<br>(basin B)" for k in range(K)], y=[f"{AXIS[k]}<br>(basin A)" for k in range(K)],
    zmid=0, colorscale=[[0, C["verm"]], [0.5, "#FFFFFF"], [1.0, C["blue"]]],
    colorbar=dict(title="M", thickness=12, len=0.85),
    text=np.round(M, 3), texttemplate="%{text}", textfont=dict(size=14),
    hovertemplate="A→B %{y} → %{x}: %{z:.3f}<extra></extra>"))
fig.update_layout(
    title=f"Least-squares map A_B ≈ A_A · M  ·  R² {r2:.3f} (relabelling only: {r2_id:.3f})  ·  "
          f"{100 * off:.0f}% of |M| off-diagonal",
    height=460, width=800, yaxis=dict(autorange="reversed"), margin=dict(l=140))
show(fig, "p6_rotation_map")

### 7 · Which models pay for it.
The Claude and Grok chains swap easy-knowledge for fluid ability; **no human tier** makes the top 12 (best rank 135 / 765).

In [8]:
sel = np.argsort(-DTH)[:12][::-1]
names = [f"<b>{models[i]}</b>" if models[i] in HUMANS else models[i] for i in sel]

fig = make_subplots(rows=1, cols=K, shared_yaxes=True, horizontal_spacing=0.03,
                    subplot_titles=[f"{AXIS[k]} ability" for k in range(K)])
for col, k in enumerate(range(K), 1):
    xs, ys = [], []
    for j, i in enumerate(sel):
        xs += [TH_A[i, k], TH_B[i, k], None]; ys += [names[j], names[j], None]
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color=C["gray"], width=2),
                             name="shift A → B", legendgroup="s", showlegend=col == 1,
                             hoverinfo="skip"), row=1, col=col)
    for tag, v, colr in [("basin A", TH_A[sel, k], C["blue"]), ("basin B", TH_B[sel, k], C["verm"])]:
        fig.add_trace(go.Scatter(x=v, y=names, mode="markers", name=tag, legendgroup=tag,
                                 showlegend=col == 1,
                                 marker=dict(color=colr, size=10, line=dict(color="white", width=1)),
                                 hovertemplate="%{y}: %{x:.2f}<extra></extra>"), row=1, col=col)
    fig.update_xaxes(title_text="θ (logits)", row=1, col=col)
fig.update_yaxes(tickfont=dict(size=10), row=1, col=1)
fig.update_layout(title="Top 12 models by ability displacement between basins",
                  height=520, width=1180, legend=dict(orientation="h", y=1.14), margin=dict(t=110, l=190))
show(fig, "p7_model_movers")

hr = sorted((int(np.sum(DTH > DTH[models.index(h)])) + 1, h) for h in HUMANS if h in models)[0]
print(f"best-ranked human tier: {hr[1]} at {hr[0]} / {len(models)}")

best-ranked human tier: Committee of Skilled Generalists at 135 / 765


### 8 · The two modes tell different capability-over-time stories.

Same models, same dates, same data. Only the axis definition differs, and the
frontier trajectory moves with it — so the mode is not a labelling detail, it
changes what "capability rose" means per axis.

In [9]:
import pandas as pd
from data import _known_release_date_by_model

dates = pd.Series(models).map(_known_release_date_by_model())
# posterior SD per basin; a model counts as MEASURED on an axis only if both
# basins pin it (the repo's SD<0.3 informed rule, applied to the stricter one)
SD = np.stack([post['theta'].values[g].reshape(-1, len(models), K).std(axis=0)
               for g in (BA, BB)]).max(axis=0)
dated = dates.notna().values & ~np.isin(models, list(HUMANS))

fig = make_subplots(rows=1, cols=K, subplot_titles=AXIS, shared_yaxes=True,
                    horizontal_spacing=0.06)
for k in range(K):
    m = dated & (SD[:, k] < 0.3)
    x = dates[m].values
    o = np.argsort(x)
    for lbl, TH, col in [('basin A', TH_A, C['blue']), ('basin B', TH_B, C['verm'])]:
        y = TH[m, k]
        fig.add_trace(go.Scatter(x=x[o], y=y[o], mode='markers', name=lbl,
                                 legendgroup=lbl, showlegend=(k == 0),
                                 marker=dict(color=col, size=5, opacity=0.55),
                                 text=np.array(models)[m][o], hovertemplate='%{text}<br>%{y:.2f}'),
                      row=1, col=k+1)
        fig.add_trace(go.Scatter(x=x[o], y=np.maximum.accumulate(y[o]), mode='lines',
                                 name=f'{lbl} frontier', legendgroup=lbl,
                                 showlegend=(k == 0), line=dict(color=col, width=2.5)),
                      row=1, col=k+1)
    r = np.corrcoef(TH_A[m, k], TH_B[m, k])[0, 1]
    fig.add_annotation(row=1, col=k+1, x=0.5, y=1.02, xref='x domain', yref='y domain',
                       showarrow=False, font=dict(size=10, color=C['dark']),
                       text=f'n={m.sum()} measured · corr(A,B)={r:.2f}')
fig.update_yaxes(title_text='ability (logits)', row=1, col=1)
fig.update_layout(height=430, width=1150, legend=dict(orientation='h', y=-0.16),
                  title='Ability timeline per axis, both basins (measured models, SD<0.3)')
show(fig, '08_timelines')

### Verdict
Two basins, 6 / 6, 32.2 nats apart; each converges on its own (r̂ ≤ 1.005), so the pooled 1.676 is a mixture, not bad sampling.
They differ by a partial rotation of the whole frame (R² 0.71, 32% off-diagonal), not by one clean axis swap.
The four 2026-07-27 arrivals that Epoch's ECI uses carry 28.0% of the squared displacement on 4% of the rows; GBAEval alone moves 4.5.
The nine new benchmarks outside the ECI are mostly inert, both FrontierMath v1 items under 0.45. Lever is scope, not more draws.